In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from imblearn.over_sampling import SMOTE
import tensorflow as tf
from tensorflow import keras
import xgboost as xgb
import matplotlib.pyplot as plt
from scipy.spatial.distance import euclidean

class LSTMXGBoostHybrid:
   def __init__(self):
      self.lstm_model = None
      self.xgb_regressor = None
      self.xgb_classifier = None
      self.scaler = StandardScaler()
      self.history = None
      
   def build_lstm_model(self, input_shape):
      # LSTM
      model = keras.Sequential([
            keras.layers.Input(shape=input_shape),
            keras.layers.LSTM(64, return_sequences=True, dropout=0.2),
            keras.layers.LSTM(32, return_sequences=False, dropout=0.2),
            keras.layers.Dense(16, activation='relu'),
            keras.layers.Dense(1, activation='sigmoid')
      ])
      
      model.compile(
            optimizer='adam', 
            loss='binary_crossentropy', 
            metrics=['accuracy']
      )
      return model
   
   def fit(self, X_train, y_train, X_val=None, y_val=None, epochs=50, batch_size=64):
      # Entrena LSTM - XGBoost
      
      # 1. Preparacion datos LSTM (formato 3D)
      X_train_lstm = X_train.reshape((X_train.shape[0], 1, X_train.shape[1]))
      if X_val is not None:
            X_val_lstm = X_val.reshape((X_val.shape[0], 1, X_val.shape[1]))
      
      # 2. Entrenar modelo LSTM
      print("Entrenando modelo LSTM...")
      self.lstm_model = self.build_lstm_model((X_train_lstm.shape[1], X_train_lstm.shape[2]))
      
      callbacks = [
            keras.callbacks.EarlyStopping(patience=10, restore_best_weights=True),
            keras.callbacks.ReduceLROnPlateau(patience=5, factor=0.5)
      ]
      
      validation_data = (X_val_lstm, y_val) if X_val is not None else None
      
      self.history = self.lstm_model.fit(
            X_train_lstm, y_train,
            validation_data=validation_data,
            epochs=epochs,
            batch_size=batch_size,
            callbacks=callbacks,
            verbose=1
      )
      
      # 3. Entrenar XGBoost Regressor (para predicción de probabilidades)
      print("Entrenando XGBoost Regressor...")
      self.xgb_regressor = xgb.XGBRegressor(
            n_estimators=300,
            max_depth=6,
            learning_rate=0.1,
            subsample=0.8,
            colsample_bytree=0.8,
            random_state=42
      )
      self.xgb_regressor.fit(X_train, y_train)
      
      # 4. Entrenar XGBoost Classifier (para clasificación binaria)
      print("Entrenando XGBoost Classifier...")
      self.xgb_classifier = xgb.XGBClassifier(
            n_estimators=500,
            max_depth=8,
            learning_rate=0.1,
            subsample=0.8,
            colsample_bytree=0.8,
            random_state=42
      )
      self.xgb_classifier.fit(X_train, y_train)
      
   def predict_hybrid(self, X_test):
      # Predicción híbrida combinando LSTM y XGBoost
      
      # Preparar datos para LSTM
      X_test_lstm = X_test.reshape((X_test.shape[0], 1, X_test.shape[1]))
      
      # 1. Obtener predicciones de ambos modelos
      lstm_pred = self.lstm_model.predict(X_test_lstm).flatten()
      xgb_reg_pred = self.xgb_regressor.predict(X_test)
      xgb_class_pred = self.xgb_classifier.predict_proba(X_test)[:, 1]
      
      # 2. Seleccionar la mejor predicción usando distancia euclidiana
      optimal_predictions = []
      
      for i in range(len(lstm_pred)):
            # Comparar las predicciones y seleccionar la óptima
            avg_pred = (lstm_pred[i] + xgb_reg_pred[i]) / 2
            optimal_predictions.append(avg_pred)
      
      optimal_predictions = np.array(optimal_predictions)
      
      # 3. Fusión probabilística, multiplicar predicción óptima por clasificación binaria
      final_predictions = optimal_predictions * xgb_class_pred
      
      # 4. Convertir a clasificación binaria
      binary_predictions = (final_predictions > 0.5).astype(int)
      
      return {
            'lstm_pred': lstm_pred,
            'xgb_reg_pred': xgb_reg_pred,
            'xgb_class_pred': xgb_class_pred,
            'optimal_pred': optimal_predictions,
            'final_pred': final_predictions,
            'binary_pred': binary_predictions
      }
   
   def predict_with_euclidean_selection(self, X_test, y_test=None):
      # Predicción usando selección por distancia euclidiana (paper)
      
      X_test_lstm = X_test.reshape((X_test.shape[0], 1, X_test.shape[1]))
      
      # Predicciones de ambos modelos
      lstm_pred = self.lstm_model.predict(X_test_lstm).flatten()
      xgb_reg_pred = self.xgb_regressor.predict(X_test)
      xgb_class_pred = self.xgb_classifier.predict_proba(X_test)[:, 1]
      
      if y_test is not None:
            # Selección basada en distancia euclidiana con valores verdaderos
            optimal_predictions = []
            for i in range(len(y_test)):
               dist_lstm = abs(lstm_pred[i] - y_test[i])
               dist_xgb = abs(xgb_reg_pred[i] - y_test[i])
               
               if dist_lstm <= dist_xgb:
                  optimal_predictions.append(lstm_pred[i])
               else:
                  optimal_predictions.append(xgb_reg_pred[i])
      else:
            # Sin valores verdaderos, promediamos
            optimal_predictions = (lstm_pred + xgb_reg_pred) / 2
      
      optimal_predictions = np.array(optimal_predictions)
      
      # Fusión probabilística
      final_predictions = optimal_predictions * xgb_class_pred
      binary_predictions = (final_predictions > 0.5).astype(int)
      
      return {
            'lstm_pred': lstm_pred,
            'xgb_reg_pred': xgb_reg_pred,
            'xgb_class_pred': xgb_class_pred,
            'optimal_pred': optimal_predictions,
            'final_pred': final_predictions,
            'binary_pred': binary_predictions
      }

# Función principal de entrenamiento y evaluación
def main():
   # Cargar y procesar datos (tu código existente)
   print("Cargando y procesando datos...")
   df = pd.read_csv("weatherAUS.csv")
   
   # Selección de características
   features = [
      'MinTemp', 'MaxTemp', 'Rainfall',
      'WindGustDir', 'WindGustSpeed',
      'WindDir9am', 'WindDir3pm',
      'WindSpeed9am', 'WindSpeed3pm',
      'Humidity9am', 'Humidity3pm',
      'Pressure9am', 'Pressure3pm',
      'Temp9am', 'Temp3pm',
      'RainToday'
   ]
   target = 'RainTomorrow'
   
   # Limpieza de datos
   df = df.drop(columns=['Evaporation', 'Sunshine', 'Cloud9am', 'Cloud3pm'])
   df = df.dropna(subset=features + [target])
   
   # Codificar variables categóricas
   wind_dir_map = {
      'N': 0.0, 'NNE': 22.5, 'NE': 45.0, 'ENE': 67.5,
      'E': 90.0, 'ESE': 112.5, 'SE': 135.0, 'SSE': 157.5,
      'S': 180.0, 'SSW': 202.5, 'SW': 225.0, 'WSW': 247.5,
      'W': 270.0, 'WNW': 292.5, 'NW': 315.0, 'NNW': 337.5
   }
   
   for col in ['WindGustDir', 'WindDir9am', 'WindDir3pm']:
      df[col] = df[col].map(wind_dir_map)
   
   df['RainToday'] = LabelEncoder().fit_transform(df['RainToday'])
   df['RainTomorrow'] = LabelEncoder().fit_transform(df['RainTomorrow'])
   
   # Separar X e y
   X = df[features].values
   y = df[target].values
   
   # Balanceo con SMOTE
   print("Aplicando SMOTE...")
   smote = SMOTE(random_state=42)
   X_res, y_res = smote.fit_resample(X, y)
   
   # Normalización
   scaler = StandardScaler()
   X_res_scaled = scaler.fit_transform(X_res)
   
   # División de datos
   X_train, X_temp, y_train, y_temp = train_test_split(
      X_res_scaled, y_res, test_size=0.4, random_state=42
   )
   X_val, X_test, y_val, y_test = train_test_split(
      X_temp, y_temp, test_size=0.5, random_state=42
   )
   
   # Crear y entrenar modelo híbrido
   print("Creando modelo híbrido LSTM + XGBoost...")
   hybrid_model = LSTMXGBoostHybrid()
   hybrid_model.fit(X_train, y_train, X_val, y_val, epochs=50, batch_size=64)
   
   # Predicciones
   print("Realizando predicciones...")
   predictions = hybrid_model.predict_with_euclidean_selection(X_test, y_test)
   
   # Evaluación
   print("\n=== RESULTADOS DEL MODELO HÍBRIDO ===")
   accuracy = accuracy_score(y_test, predictions['binary_pred'])
   print(f"Accuracy del modelo híbrido: {accuracy*100:.2f}%")
   
   print("\nReporte de clasificación:")
   print(classification_report(y_test, predictions['binary_pred']))
   
   # Comparación con modelos individuales
   print("\n=== COMPARACIÓN CON MODELOS INDIVIDUALES ===")
   
   # LSTM solo
   X_test_lstm = X_test.reshape((X_test.shape[0], 1, X_test.shape[1]))
   lstm_pred_binary = (hybrid_model.lstm_model.predict(X_test_lstm).flatten() > 0.5).astype(int)
   lstm_accuracy = accuracy_score(y_test, lstm_pred_binary)
   print(f"Accuracy LSTM solo: {lstm_accuracy*100:.2f}%")

   print("\n=== RESULTADOS DEL MODELO LSTM SOLO ===")
   print(f"Accuracy LSTM solo: {lstm_accuracy*100:.2f}%")
   print("\nReporte de clasificación:")
   print(classification_report(y_test, lstm_pred_binary, target_names=["No Llovera", "Si Llovera"]))

   
   # XGBoost solo
   xgb_pred_binary = hybrid_model.xgb_classifier.predict(X_test)
   xgb_accuracy = accuracy_score(y_test, xgb_pred_binary)
   print(f"Accuracy XGBoost solo: {xgb_accuracy*100:.2f}%")
   
   print("\n=== RESULTADOS DEL MODELO XGBOOST SOLO ===")
   print(f"Accuracy XGBoost solo: {xgb_accuracy*100:.2f}%")
   print("\nReporte de clasificación:")
   print(classification_report(y_test, xgb_pred_binary, target_names=["No Llovera", "Si Llovera"]))

   
   # Visualización
   plt.figure(figsize=(15, 5))
   
   # Curvas de entrenamiento LSTM
   plt.subplot(1, 3, 1)
   plt.plot(hybrid_model.history.history['loss'], label='Train Loss')
   if 'val_loss' in hybrid_model.history.history:
      plt.plot(hybrid_model.history.history['val_loss'], label='Val Loss')
   plt.title('Curvas de Pérdida LSTM')
   plt.legend()
   
   # Comparación de accuracies
   plt.subplot(1, 3, 2)
   models = ['LSTM', 'XGBoost', 'Híbrido']
   accuracies = [lstm_accuracy, xgb_accuracy, accuracy]
   bars = plt.bar(models, accuracies, color=['blue', 'green', 'red'])
   plt.title('Comparación de Accuracies')
   plt.ylabel('Accuracy')
   
   # Agregar valores en las barras
   for bar, acc in zip(bars, accuracies):
      plt.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01, 
               f'{acc:.3f}', ha='center', va='bottom')
   
   # Matriz de confusión del modelo híbrido
   plt.subplot(1, 3, 3)
   cm = confusion_matrix(y_test, predictions['binary_pred'])
   plt.imshow(cm, interpolation='nearest', cmap=plt.cm.Blues)
   plt.title('Matriz de Confusión\n(Modelo Híbrido)')
   plt.colorbar()
   
   # Etiquetas
   tick_marks = np.arange(2)
   plt.xticks(tick_marks, ['No Rain', 'Rain'])
   plt.yticks(tick_marks, ['No Rain', 'Rain'])
   
   # Agregar números en la matriz
   for i in range(2):
      for j in range(2):
            plt.text(j, i, str(cm[i, j]), ha='center', va='center')
   
   plt.tight_layout()
   plt.show()
   
   return hybrid_model, predictions

if __name__ == "__main__":
   # Ejecutar entrenamiento y evaluación
   model, results = main()

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import classification_report, confusion_matrix, mean_squared_error
from imblearn.over_sampling import SMOTE
import tensorflow as tf
from tensorflow import keras
import matplotlib.pyplot as plt

# === 1. Cargar y preparar datos ===
df = pd.read_csv("weatherAUS.csv")

# Columnas elegidas (mismo enfoque que paper)
features = [
    'MinTemp', 'MaxTemp', 'Rainfall',
    'WindGustDir', 'WindGustSpeed',
    'WindDir9am', 'WindDir3pm',
    'WindSpeed9am', 'WindSpeed3pm',
    'Humidity9am', 'Humidity3pm',
    'Pressure9am', 'Pressure3pm',
    'Temp9am', 'Temp3pm',
    'RainToday'
]
target = 'RainTomorrow'

# Eliminar columnas con >40% NaN
df = df.drop(columns=['Evaporation', 'Sunshine', 'Cloud9am', 'Cloud3pm'])

# Eliminar NaNs
df = df.dropna(subset=features + [target])

# Codificación
wind_dir_map = {
    'N': 0.0, 'NNE': 22.5, 'NE': 45.0, 'ENE': 67.5,
    'E': 90.0, 'ESE': 112.5, 'SE': 135.0, 'SSE': 157.5,
    'S': 180.0, 'SSW': 202.5, 'SW': 225.0, 'WSW': 247.5,
    'W': 270.0, 'WNW': 292.5, 'NW': 315.0, 'NNW': 337.5
}
for col in ['WindGustDir', 'WindDir9am', 'WindDir3pm']:
    df[col] = df[col].map(wind_dir_map)

df['RainToday'] = LabelEncoder().fit_transform(df['RainToday'])
df['RainTomorrow'] = LabelEncoder().fit_transform(df['RainTomorrow'])

# Verificar que tenemos solo 2 clases
print("Verificación de clases:")
print(f"RainTomorrow único: {sorted(df['RainTomorrow'].unique())}")
print(f"Distribución: {df['RainTomorrow'].value_counts().sort_index()}")

# === 2. División de datos y normalización ===
X = df[features]
y = df[target]

# SMOTE
smote = SMOTE(random_state=42)
X_res, y_res = smote.fit_resample(X, y)

print(f"\nDespués de SMOTE:")
print(f"y_res único: {sorted(np.unique(y_res))}")
print(f"Distribución: {np.bincount(y_res)}")

# Normalización
scaler = StandardScaler()
X_res_scaled = scaler.fit_transform(X_res)

# Crear secuencias temporales CORRECTAMENTE
def create_sequences(X, y, window=3):
    X_seq, y_seq = [], []
    for i in range(len(X) - window):
        X_seq.append(X[i:i+window])
        y_seq.append(y[i+window])
    return np.array(X_seq), np.array(y_seq)

# Crear secuencias
X_seq, y_seq = create_sequences(X_res_scaled, y_res, window=3)

print(f"\nDespués de crear secuencias:")
print(f"X_seq shape: {X_seq.shape}")
print(f"y_seq único: {sorted(np.unique(y_seq))}")
print(f"y_seq distribución: {np.bincount(y_seq)}")

# División train/test
X_train, X_test, y_train, y_test = train_test_split(X_seq, y_seq, test_size=0.2, random_state=42)

print(f"\nDatos finales:")
print(f"X_train shape: {X_train.shape}")
print(f"y_train único: {sorted(np.unique(y_train))}")
print(f"y_test único: {sorted(np.unique(y_test))}")

# === Transformer ===
def transformer_encoder(inputs, head_size, num_heads, ff_dim, dropout=0):
    # Multi-Head Attention
    x = keras.layers.MultiHeadAttention(
        key_dim=head_size, 
        num_heads=num_heads, 
        dropout=dropout
    )(inputs, inputs)
    x = keras.layers.Dropout(dropout)(x)
    x = keras.layers.LayerNormalization(epsilon=1e-6)(x + inputs)

    # Feed Forward Network
    x_ff = keras.layers.Conv1D(filters=ff_dim, kernel_size=1, activation="relu")(x)
    x_ff = keras.layers.Dropout(dropout)(x_ff)
    x_ff = keras.layers.Conv1D(filters=inputs.shape[-1], kernel_size=1)(x_ff)
    return keras.layers.LayerNormalization(epsilon=1e-6)(x + x_ff)

# Construir modelo
inputs = keras.Input(shape=(X_train.shape[1], X_train.shape[2]))
x = transformer_encoder(inputs, head_size=32, num_heads=2, ff_dim=64, dropout=0.1)
x = keras.layers.GlobalAveragePooling1D()(x)
x = keras.layers.Dense(32, activation="relu")(x)
x = keras.layers.Dropout(0.2)(x)
x = keras.layers.Dense(1, activation="sigmoid")(x)

model = keras.Model(inputs, x)
model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=0.001),
    loss="binary_crossentropy", 
    metrics=["accuracy"]
)
model.summary()

# === Entrenamiento ===
callbacks = [
    keras.callbacks.EarlyStopping(patience=10, restore_best_weights=True),
    keras.callbacks.ReduceLROnPlateau(factor=0.5, patience=5),
    keras.callbacks.ModelCheckpoint("best_transformer_model.h5", save_best_only=True)
]

history = model.fit(
    X_train, y_train,
    validation_data=(X_test, y_test),
    epochs=50,
    batch_size=64,
    callbacks=callbacks,
    verbose=1
)

# === 5. Evaluación ===
loss, acc = model.evaluate(X_test, y_test)
print(f"\nAccuracy en test: {acc * 100:.2f}%")

# === 6. Métricas avanzadas ===
y_pred_prob = model.predict(X_test)
y_pred = (y_pred_prob > 0.5).astype(int).flatten()

# Verificación final
print(f"\nVerificación final:")
print(f"y_test único: {sorted(np.unique(y_test))}")
print(f"y_pred único: {sorted(np.unique(y_pred))}")

print("\nMatriz de Confusión:")
print(confusion_matrix(y_test, y_pred))

print("\nReporte de Clasificación:")
print(classification_report(y_test, y_pred, target_names=["No Lloverá", "Sí Lloverá"]))

rmse = np.sqrt(mean_squared_error(y_test, y_pred_prob))
print(f"\nRMSE: {rmse:.4f}")

# === 7. Visualización ===
plt.figure(figsize=(12, 4))

plt.subplot(1, 2, 1)
plt.plot(history.history['loss'], label='Training Loss')
plt.plot(history.history['val_loss'], label='Validation Loss')
plt.title('Model Loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()

plt.subplot(1, 2, 2)
plt.plot(history.history['accuracy'], label='Training Accuracy')
plt.plot(history.history['val_accuracy'], label='Validation Accuracy')
plt.title('Model Accuracy')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.legend()

plt.tight_layout()
plt.show()